In [0]:
dbutils.widgets.text("catalog_name","resume_batch3","")
catalog_name=dbutils.widgets.get("catalog_name")

In [0]:
from pyspark.sql import functions as F

In [0]:
%py
spark.sql(f"""create table if not exists {catalog_name}.gold.company_jd
as
select *,monotonically_increasing_id() as company_jd_id,concat_ws(title,experience,skills,'|') as merged_description from resume_batch3.silver.company_jd""")

In [0]:
is_table_company_jd_exists=spark.catalog.tableExists(f'{catalog_name}.gold.company_jd')

In [0]:
if is_table_company_jd_exists:
    df = (
        spark.read.table(f"{catalog_name}.silver.company_jd")
        .withColumn(
            "merged_description",
            F.concat_ws(
                F.lit("|"), F.col("title"), F.col("experience"), F.col("skills")
            ),
        )
        .withColumn("company_jd_id", F.monotonically_increasing_id())
    )
    df.createOrReplaceTempView("temp_company_vw")
    spark.sql(
        f"""merge into {catalog_name}.gold.company_jd t using temp_company_vw s on t.path
 = s.path when matched then update set * when not matched then insert *"""
    )

In [0]:
%py
spark.sql(f"""create table if not exists {catalog_name}.gold.resume_data
as
select *,monotonically_increasing_id() as resume_id,concat_ws(title,experience,skills,'|') as merged_description from resume_batch3.silver.resume_data""")

In [0]:
is_table_resume_data_exists=spark.catalog.tableExists(f'{catalog_name}.gold.resume_data')

In [0]:

if is_table_resume_data_exists:
    df = spark.read.table(f"{catalog_name}.silver.resume_data").withColumn("merged_description",F.concat_ws(F.lit('|'),F.col("title"),F.col("experience"),F.col("skills"))).withColumn("resume_id",F.monotonically_increasing_id())
    df.createOrReplaceTempView("temp_resume_data_vw")
    spark.sql(f"""merge into {catalog_name}.gold.resume_data t using temp_resume_data_vw s on t.path
 = s.path when matched then update set * when not matched then insert *"""
    )

In [0]:
%py
spark.sql(f"""ALTER TABLE {catalog_name}.gold.company_jd 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)""")


In [0]:
%py
spark.sql(f"""ALTER TABLE {catalog_name}.gold.resume_data 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)""")
